# 2. Detector imaging, noise, masks, and likelihoods

This notebook turns a noiseless lensed surface-brightness model into a detector observation. Project Akira uses normalized, odd-sized PSF kernels and zero-padded linear convolution, so flux cannot wrap from one image edge to the opposite edge.

For independent Gaussian noise, the normalized image likelihood is

$$\log\mathcal L=-\frac12\sum_{j\in\mathrm{unmasked}}\left[\frac{(d_j-m_j)^2}{\sigma_j^2}+\log(2\pi\sigma_j^2)\right].$$

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from akira import (
    CartesianGrid, Gaussian, SIS, gaussian_log_likelihood, gaussian_psf,
    render_sis_gaussian, simulate_imaging,
)
from akira.imaging import convolve_psf
from akira.likelihood import chi_squared, normalized_residuals

In [ ]:
grid = CartesianGrid(shape=(120, 120), pixel_scale=0.035, supersampling=2)
lens = SIS(einstein_radius=1.0)
source = Gaussian(
    intensity=1.0, sigma=0.09, centre=(0.08, -0.04), q=0.72, phi=0.35
)
psf = gaussian_psf(sigma_pixels=1.0, size=9)

unconvolved = render_sis_gaussian(grid, lens, source)
convolved = convolve_psf(unconvolved, psf.kernel)
print(f'PSF sum: {float(psf.kernel.sum()):.15f}')
print(f'PSF shape: {psf.kernel.shape}')

## Create a mask and reproducible noisy observation

A mask value of `True` means that the pixel is excluded from likelihood evaluation. The full detector image is still retained. Noise requires an explicit seed; running the same simulation with the same seed returns identical data.

In [ ]:
coordinates = np.asarray(grid.coordinates())
radius = np.linalg.norm(coordinates, axis=-1)
mask = radius > 1.65

observation = simulate_imaging(
    unconvolved_image=unconvolved,
    psf=psf,
    noise_sigma=0.02,
    pixel_scale=grid.pixel_scale,
    seed=42,
    mask=mask,
)
repeat = simulate_imaging(unconvolved, psf, 0.02, grid.pixel_scale, seed=42, mask=mask)
np.testing.assert_array_equal(observation.data, repeat.data)
print('Repeated seeded simulation is exactly reproducible.')

In [ ]:
residual = np.asarray(observation.data - convolved)
normalized = np.asarray(normalized_residuals(observation.data, convolved, observation.noise_map))

fig, axes = plt.subplots(2, 3, figsize=(13, 8), constrained_layout=True)
panels = [
    (psf.kernel, 'Normalized PSF'),
    (unconvolved, 'Unconvolved lens model'),
    (convolved, 'PSF-convolved model'),
    (observation.data, 'Noisy observation'),
    (residual, 'Residual: data - model'),
    (normalized, 'Normalized residual'),
]
for ax, (image, title) in zip(axes.flat, panels):
    display = ax.imshow(np.asarray(image), origin='lower', cmap='magma')
    ax.set_title(title)
    fig.colorbar(display, ax=ax, shrink=0.8)
plt.show()

## Evaluate the model statistically

The data were generated from this model plus Gaussian noise, so the reduced chi-squared should fluctuate near one over the unmasked pixels.

In [ ]:
log_likelihood = gaussian_log_likelihood(
    observation.data, convolved, observation.noise_map, observation.mask
)
chi2 = chi_squared(observation.data, convolved, observation.noise_map, observation.mask)
n_unmasked = int((~np.asarray(observation.mask)).sum())

print(f'Unmasked pixels:     {n_unmasked}')
print(f'Log likelihood:      {float(log_likelihood):.3f}')
print(f'Chi-squared:         {float(chi2):.3f}')
print(f'Reduced chi-squared: {float(chi2) / n_unmasked:.3f}')